## The Plastic Additive LCA Data Finder Personal Dashboard (PLADLCADF_PDB)

This notebook is designed to help you sort through and find data for additives in your plastics using the Plastic Additive Life Cycle Assessment Data Finder (PLAD_LCA_DF). It takes the output from the latest version of the PLAD_LCA_DF and allows you to filter the returns to find additives with known useage in a given context (industry and function) for particular polymers and find availble data. You can search the database using part 1, part 2, or both parts of this notebook. Part 1 allows you to search for an additive based on the use case, while part 2 allows you to search by a search string for a specific additive. We also visualize the results for you so you have some eacy to use figures on the trends in additives and data availbility for your study. 

The PLAD_LCA_DF links state-of-the-art knowledge from both the Plastchem database (2024) and the UNEP chemicals in plastics database (2023) to additives coverage in the ecoinvent 3.9.1 & 3.10, LCA for Experts 2023, and CarbonMinds 2022 LCA databases following the approach developed in Logan et al. (2024). To read more about this method please refer to Logan et al (2024).

The Plastic Additive LCA Data Finder Personal Dashboard (PLADLCADF_PDB) version 1.0 © 2025 by Heather Margaret Logan is licensed under Creative Commons Attribution-NonCommercial-ShareAlike (CC BY-NC-SA) 4.0 International. This means you are welcome to remix and adapt this work; however, it cannot be used for commercial purposes, you must credit the source, and license the resulting work under the same CC BY-NC-SA 4.0 international license.

To cite the method for this tool please cite: Logan, H., S. DeMeester, T.F. Astrup, A. Damgaard. 2024. Additive Inclusion in Plastic Life Cycle Assessments, Part II: Review of additive inventory data trends and availability. Journal of Industrial Ecology. https://doi.org/10.1111/jiec.13534

To cite the PLCA LCA DF v 1.0 and all subtools please cite: Logan, H. 2024. Plastic Additives (PLAD) Life Cycle Assessment (LCA) Data Finder (DF) & Tools. Version 1.0. https://github.com/hmlogan/PLADLCADF

So we start by importing the packages needed to run the code

In [1]:
import dash
from dash import html, dcc, Input, Output, State, ctx, callback_context, dash_table
from dash.exceptions import PreventUpdate 
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import re
import requests

This cell retrieves the file from the github repository and turns it into stored data on your personal computer. 

In [2]:
url = ("https://github.com/hmlogan/PLADLCADF/raw/main/PLAD_LCADF_ADDDB_v1.0.xlsx")

response = requests.get(url)
with open("temp.xlsx", "wb") as f:
    f.write(response.content)

path = "temp.xlsx"
sheet_name = "plad"
plad_lca_db_df = pd.read_excel(path, sheet_name = sheet_name)
# Remove the hashtag from the line below if you want to check and be sure the data imported correctly. 
#plad_lca_db_df

This cell sets up the filters that we use on the database.

In [3]:
polymer_columns = ['ABS', 'RUBBERS', 'PVC', 'COPOLYMER', 'EPOXIES', 'HIPS', 'PS', 'PC',
                   'PMMA', 'PA', 'POLYESTERS', 'POM', 'PUR', 'WEEE', 'PP', 'HDPE', 'PET', 'LDPE']

industry_columns = ['AGRICULTURE', 'AUTOMOTIVE', 'BUILDING & CONSTRUCTION', 'ELECTRICAL',
                    'ELECTRONIC EQUIPMENT', 'FOOD-CONTACT PLASTICS', 'FURNITURE',
                    'HOUSEHOLD ITEMS', 'HYGIENE', 'MEDICAL ITEMS', 'PACKAGING', 'TEXTILES', 'TOYS']

additive_function_columns = ['ADHESION-IMPROVING AGENTS', 'ANTI_FOG AGENTS', 'ANTIFRICTION AGENTS',
                             'ANTIOXIDANTS', 'ANTISTATIC AGENT', 'BLOWING AGENT', 'COLORANTS',
                             'DEGRADATION PRODUCT', 'FILLERS', 'FLAME RETARDANTS',
                             'FRAGRANCE AGENTS', 'LIGHT STABILIZER', 'LUBRICANT', 'NOT REPORTED',
                             'OTHER', 'OTHER PROCESSING AIDS', 'OTHER STABILIZER',
                             'OTHER SURFACE PROTECTORS', 'PIGMENT', 'PLASTICIZER',
                             'PRODUCTION AGENTS', 'REINFORCEMENT', 'SURFACE PROTECTANT',
                             'THERMOSTABILIZERS', 'UV STABILIZER']


database_columns = ['lca4e_23', 'lca4e_24', 'cm_22', 'cm_24',
                    'ei_391_uao', 'ei_391_co', 'ei_391_apos', 'ei_391_conseq',
                    'ei_310_uao', 'ei_310_co', 'ei_310_apos', 'ei_310_conseq']


## Part 1 
### This section is designed to help you sort and filter the database to return the additives which have known uses for your product under study. You can add or remove filters by selecting the check boxes and then graphs are generated that you can export on the total number of additives available per function and industry for each polymer. In addition, you will find graphs on the watch list classification for each additive according to PLASTCHEM. Once you have filtered the database as you would like for your plastic you can move onto part 3 to export the table. If you would like to search for a specific additive to add to your exported table, go to part 2.


In [62]:
df = plad_lca_db_df
filtered_result_df = pd.DataFrame

app = dash.Dash(__name__)

app.layout = html.Div([
    html.H3("Filter Additives by Polymer and Industry"),

    html.Label("Select Polymer(s):"),
    dcc.Checklist(
        id="polymer-column-filter",
        options=[{'label': col, 'value': col} for col in polymer_columns],
        value=['PET'],
        inline=True
    ),
    html.Label("Select Industry(ies):"),
    dcc.Checklist(
        id="industry-column-filter",
        options=[{'label': col, 'value': col} for col in industry_columns],
        value=[],
        inline=True
    ),
    html.Label("Select Additive Function(s)):"),
    dcc.Checklist(
        id="additive-function-column-filter",
        options=[{'label': col, 'value': col} for col in additive_function_columns],
        value=[],
        inline=True
    ),
    html.Label("Select LCA Database):"),
    dcc.Checklist(
        id="database-column-filter",
        options=[{'label': col, 'value': col} for col in database_columns],
        value=[],
        inline=True,
    ),
    html.Label("Select Hazard List(s):"),
    dcc.Checklist(
        id = "hazard-filter",
        options = [{'label': row, 'value': row} for row in df['PlastChem CASRN registered on a watch list'].unique()],
        inline=True,
    ),

    html.Div(id="filtered-table"),
    html.Hr(),
    html.Div(id="function-summary")
])


@app.callback(
    Output("filtered-table", "children"),
    Output("function-summary", "children"),
    Input("polymer-column-filter", "value"),
    Input("industry-column-filter", "value"), 
    Input("additive-function-column-filter", "value"),
    Input("database-column-filter", "value"),
    Input("hazard-filter", "value")
)
def update_tables(selected_polymers, selected_industries, selected_additive_function, selected_database, selected_hazard):
    filtered_df = df.copy()

    if selected_polymers:
        polymer_mask = filtered_df[selected_polymers].sum(axis=1) >= 1
        filtered_df = filtered_df[polymer_mask]

    if selected_industries:
        industry_mask = filtered_df[selected_industries].sum(axis=1) >= 1
        filtered_df = filtered_df[industry_mask]
    
    if selected_additive_function:
        additive_function_mask = filtered_df[selected_additive_function].sum(axis=1) >= 1
        filtered_df = filtered_df[additive_function_mask]
    
    if selected_database:
        database_mask = filtered_df[selected_database].sum(axis=1) >= 1
        filtered_df = filtered_df[database_mask]

    if selected_hazard:
        filtered_df = filtered_df[filtered_df['PlastChem CASRN registered on a watch list'].isin(selected_hazard)]
    
    base_columns = ['Name', 'CASRN','PlastChem CASRN registered on a watch list'] 
    display_columns = base_columns + selected_polymers + selected_industries + selected_additive_function
    
    global filtered_result_df
    filtered_result_df = filtered_df.copy()

    if filtered_df.empty:
        return html.Div("No matching results found."), html.Div()

    main_table = html.Div([
        html.H4("Filtered Additives Table"),
        dash_table.DataTable(
            columns=[{"name": col, "id": col} for col in display_columns],
            data=filtered_df[display_columns].to_dict("records"),
            style_cell={
                'whiteSpace': 'normal',
                'textAlign': 'left',
                'height': 'auto',
                'width' : 'auto'
            }
        )
    ])

    charts = []
    if selected_polymers and selected_industries and selected_additive_function:
        for polymer in selected_polymers:
            polymer_df = filtered_df[filtered_df[polymer] == 1]

            summary_records = []
            for func in selected_additive_function:
                func_df = polymer_df[polymer_df[func] == 1]
                for ind in selected_industries:
                    count = func_df[ind].sum()
                    if count > 0:
                        summary_records.append({
                            'Function': func,
                            'Industry': ind,
                            'Count': count
                        })

            if summary_records:
                summary_df = pd.DataFrame(summary_records)
                fig = px.bar(
                    summary_df,
                    x='Function',
                    y='Count',
                    color='Industry',
                    barmode='group',
                    title=f"Additives availble per selected function for use in {polymer} for selected industries"
                )
                charts.append(dcc.Graph(figure=fig))

    return main_table, html.Div(charts)



if __name__ == "__main__":
    app.run(debug=True, port=8052)


## Part 2: 
### Search here for specific additives you would like to add to the exported data frame. You can either search by name or CAS-RN. Each time you click save the filtered data will be saved as a data frame. 


If the table isn’t updating even though you have entered text, you can click and unclick a check box or click into the other search bar and the table should update. Remember every time you hit Add you add all visible columns to the data frame. If you make a mistake and add rows you can simply restart this cell, and it will clear out any rows you have added using the search string functions. 

In [5]:
df_out=pd.DataFrame()
app = dash.Dash(__name__)
server = app.server

app.layout = html.Div([
    html.H3("Filter Additives"),

    html.Label("Search Additive Name:"),
    dcc.Input(id='name-search-input', type='text', placeholder='Enter name...', debounce=True),

    html.Label("Search CASRN:"),
    dcc.Input(id='CASRN-search-input', type='text', placeholder='Enter CASRN...', debounce=True),

    html.Label("Select LCA Database(s):"),
    dcc.Checklist(
        id="database-column-filter",
        options=[{'label': col, 'value': col} for col in database_columns],
        value=[],
        inline=True
    ),

    html.Br(),
    html.Button('Add all visible rows to personal database', id='add-button', n_clicks=0),
    html.Div(id='filtered-table'),
    html.Div(id='save-message'),

    dcc.Store(id='filtered-data-store')
])


@app.callback(
    Output("filtered-table", "children"),
    Output("filtered-data-store", "data"),
    Input("name-search-input", "value"),
    Input("CASRN-search-input", "value"),
    Input("database-column-filter", "value")
)
def update_table(name_search_value, CASRN_search_value,
                 selected_databases):

    filtered = df.copy()

    if name_search_value:
        filtered = filtered[
            (filtered['Name'] == name_search_value) |
            (filtered['Name'].str.contains(name_search_value, case=False, na=False))
        ]
    if CASRN_search_value:
        filtered = filtered[
            (filtered['CASRN'] == CASRN_search_value) |
            (filtered['CASRN'].str.contains(CASRN_search_value, case=False, na=False))
        ]
  
    if selected_databases:
        filtered = filtered[filtered[selected_databases].sum(axis=1) >= 1]

    if filtered.empty:
        return html.Div("No matching results found."), html.Div(), None

    base_columns = ['Name', 'CASRN', 'PlastChem CASRN registered on a watch list']
    all_selected_columns = selected_databases
    display_columns = base_columns + all_selected_columns

    main_table = dash_table.DataTable(
        columns=[{'name': col, 'id': col} for col in display_columns],
        data=filtered[display_columns].to_dict("records"),
        style_cell={'whiteSpace': 'normal', 'textAlign': 'left'}
    )

    return html.Div(main_table), filtered.to_json(date_format='iso', orient='split')


@app.callback(
    Output("save-message", "children"),
    Input("add-button", "n_clicks"),
    State("filtered-data-store", "data")
)
def save_data(n_clicks, json_data):
    if n_clicks > 0 and json_data:
        global df_out
        if df_out.empty:
            df_out = pd.read_json(json_data, orient='split')
        if not df_out.empty:
            new_entry = pd.read_json(json_data, orient='split')
            df_out = pd.concat([df_out, new_entry], ignore_index=True)
            df_out = df_out.drop_duplicates()
               
        return "Filtered results saved to 'df_out'."
    return ""

if __name__ == '__main__':
    app.run(debug=True, port=8051)


## Part 3: Exporting and Visualizing your personal database.
### This step allows you to check, visualize, and export your personal additive database to use in your research outside of this notebook. There is a graph showing you the database with the best data coverage for your study, the function of additives per polymer, and the watchlist status of your database. 


In [8]:
# Part 1 output: Remove the hashtag on the line below to view the dataframe in below this cell. Only try to view one DF at a time. 
#filtered_result_df

# Part 2 output: Remove the hashtag on the line below to view the dataframe in below this cell. Only try to view one DF at a time. 
#df_out

In [63]:
if df_out.empty: 
    combined_df = filtered_result_df
    combined_df = combined_df.drop_duplicates()
else: 
    combined_df = pd.concat([filtered_result_df, df_out], ignore_index=True)
    combined_df = combined_df.drop_duplicates()

combined_df

,Source,Unique ID,CASRN,Name,PlastChem CASRN registered on a watch list,Regulation?,Function (UNEP & PlastChem),ADHESION-IMPROVING AGENTS,ANTI_FOG AGENTS,ANTIFRICTION AGENTS,...,cm_22,cm_24,ei_391_uao,ei_391_co,ei_391_apos,ei_391_conseq,ei_310_uao,ei_310_co,ei_310_apos,ei_310_conseq
223,UNEP,3105,101239-80-9,Carbon black,No Data,1,"PIGMENT AGENT,ANTISTATIC,UV/LIGHT STABILISER, ...",0,0,0,...,0,0,1,1,1,1,1,1,1,1
20356,UNEP,3968,72608-12-9,Carbonic acid calcium salt (1:1),No Data,0,"PIGMENT AGENT, ANTIOXIDANT, BIOCIDE, BLOWING A...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
24505,UNEP,11973,94340-28-0,"2-Propanol, titanium(4+) salt (4:1)",No Data,0,", CATALYST, COLORANT, CROSSLINKING AGENT, FILL...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
25266,UNEP,14569,99400-01-8,"Sulfuric acid, calcium salt (1:1)",No Data,0,"PIGMENT AGENT, BIOCIDE, COLORANT, CROSSLINKING...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
25327,PLASTCHEM,6,100-21-0,Terephthalic acid,Red_list,0,", COLORANT, FILLER, INTERMEDIATES, LUBRICANT, ...",0,0,0,...,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41525,PLASTCHEM,16204,1317-61-9,Iron oxide (Fe3O4),Red_list,0,"PIGMENT AGENT, ANTIOXIDANT, BIOCIDE, COLORANT,...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
41528,PLASTCHEM,16207,1330-20-7,m-Xylene,Red_list,1,", ANTISTATIC AGENT, BIOCIDE, CATALYST, COLORAN...",0,0,0,...,1,1,1,1,1,1,0,0,0,0
41534,PLASTCHEM,16213,147-14-8,Copper(II) phthalocyanine,Red_list,0,"PIGMENT AGENT, ANTISTATIC AGENT, COLORANT, CRO...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
41632,PLASTCHEM,16311,7786-30-3,Magnesium Chloride,Grey_list,0,", BIOCIDE, CATALYST, COLORANT, FILLER, INTERME...",0,0,0,...,0,0,1,1,1,1,1,1,1,1


In [ ]:

# 1. Function Frequency by Industry Application
polymer_cols = ['ABS', 'RUBBERS', 'PVC', 'COPOLYMER', 'EPOXIES', 'HIPS', 'PS', 'PC',
                   'PMMA', 'PA', 'POLYESTERS', 'POM', 'PUR', 'WEEE', 'PP', 'HDPE', 'PET', 'LDPE']

industry_cols = ['AGRICULTURE', 'AUTOMOTIVE', 'BUILDING & CONSTRUCTION', 'ELECTRICAL',
                    'ELECTRONIC EQUIPMENT', 'FOOD-CONTACT PLASTICS', 'FURNITURE',
                    'HOUSEHOLD ITEMS', 'HYGIENE', 'MEDICAL ITEMS', 'PACKAGING', 'TEXTILES', 'TOYS']

additive_function_cols = ['ADHESION-IMPROVING AGENTS', 'ANTI_FOG AGENTS', 'ANTIFRICTION AGENTS',
                             'ANTIOXIDANTS', 'ANTISTATIC AGENT', 'BLOWING AGENT', 'COLORANTS',
                             'DEGRADATION PRODUCT', 'FILLERS', 'FLAME RETARDANTS',
                             'FRAGRANCE AGENTS', 'LIGHT STABILIZER', 'LUBRICANT', 'NOT REPORTED',
                             'OTHER', 'OTHER PROCESSING AIDS', 'OTHER STABILIZER',
                             'OTHER SURFACE PROTECTORS', 'PIGMENT', 'PLASTICIZER',
                             'PRODUCTION AGENTS', 'REINFORCEMENT', 'SURFACE PROTECTANT',
                             'THERMOSTABILIZERS', 'UV STABILIZER']


# database_cols = ['lca4e_23', 'lca4e_24', 'cm_22', 'cm_24',
#                     'ei_391_uao', 'ei_391_co', 'ei_391_apos', 'ei_391_conseq',
#                     'ei_310_uao', 'ei_310_co', 'ei_310_apos', 'ei_310_conseq']
database_cols = [ 'lca4e_24', 'cm_24','ei_310_co']


# Fig 1 

db_melt = combined_df[['Name'] + database_cols].melt(id_vars='Name', 
                                                     value_vars=database_cols,
                                                     var_name='Database',
                                                     value_name='Present')

# Filter to only present substances
db_melt = db_melt[db_melt['Present'].notna() & (db_melt['Present'] != 0)]

# Count occurrences per substance per database
db_counts = db_melt.groupby(['Database', 'Name']).size().reset_index(name='Count')

# Plot stacked bar
fig1_new = px.bar(db_counts, x='Database', y='Count', color_continuous_scale='Reds',
                  title='Additive Availiblity in Databases',
                  labels={'Count': 'Substance Count'}, 
                  )
fig1_new.update_layout(barmode='stack')
fig1_new.show()

#Fig 2

for polymer in polymer_cols:
    combined_df[polymer] = combined_df['Polymer (UNEP)'].str.contains(polymer).astype(int)

poly_heat_df = combined_df[['Name'] + polymer_cols]
fig2 = px.imshow(poly_heat_df.set_index('Name'), color_continuous_scale='Blues',
                 title='Additive-Polymer Association Heatmap', aspect='auto')
fig2.show()

#Figure 3 

selected_polymer = 'PP' #change to the polymer you would prefer to focus on


for polymer in polymer_cols:
    combined_df[polymer] = combined_df['Polymer (UNEP)'].str.contains(polymer).astype(int)

for function in additive_function_cols:
    combined_df[function] = combined_df['Function (UNEP & PlastChem)'].str.contains(function).astype(int)

filtered_df = combined_df[combined_df[selected_polymer] == 1]


function_melt = filtered_df.melt(
    id_vars=['CASRN', 'PlastChem CASRN registered on a watch list'],
    value_vars=additive_function_cols,
    var_name='Function',
    value_name='Present'
)

function_melt = function_melt[function_melt['Present'] == 1]

heatmap_data = function_melt.groupby(
    ['PlastChem CASRN registered on a watch list', 'Function']
)['CASRN'].nunique().reset_index(name='Substance Count')

heatmap_matrix = heatmap_data.pivot(
    index='PlastChem CASRN registered on a watch list',
    columns='Function',
    values='Substance Count'
).fillna(0)

fig3 = px.imshow(
    heatmap_matrix,
    labels={'color': 'Substance Count'},
    color_continuous_scale='Reds',
    title=f'Additive Count by Watch List Status and Function for {selected_polymer}'
)
fig3.update_layout(
    yaxis_title='Watch List Status',
    xaxis_title='Function',
)
fig3.show()


# Fig 4

func_polymer_matrix = pd.DataFrame(index=additive_function_cols, columns=polymer_cols).infer_objects(copy=False)


for func in additive_function_cols:
    for polymer in polymer_cols:
        count = combined_df[(combined_df[func] == 1) & (combined_df[polymer] == 1)].shape[0]
        func_polymer_matrix.loc[func, polymer] = count


func_polymer_matrix = func_polymer_matrix.astype(int)


fig4 = px.imshow(func_polymer_matrix,
                      labels=dict(x='Polymer', y='Function', color='Substance Count'),
                      x=polymer_cols,
                      y=additive_function_cols,
                      color_continuous_scale='Blues',
                      title='Substance Count per Function and Polymer Type', 
                      aspect='auto')
fig4.update_layout(yaxis=dict(tickmode='array', tickvals=list(range(len(additive_function_cols))),
                                   ticktext=additive_function_cols))
fig4.show()




When you are ready to export your database, you can simply run the cell below and it will save the file as an excel for you to access later. Don't forget to export SVGs you want to keep while you are here otherwise you will have to run the program again of edit the code to read your excel and regenerate the figures. 

In [ ]:
combined_df.to_excel('PLADLCADF_PDB.xlsx', index=False)